# Load data

In [15]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("Loading data...")
# Master
df_pduct = pd.read_parquet('../dataset/cleaned/products.parquet')
df_cus = pd.read_parquet('../dataset/cleaned/customers.parquet')
df_pmot = pd.read_parquet('../dataset/cleaned/promotions.parquet')
df_geo = pd.read_parquet('../dataset/cleaned/geography.parquet')

# Transaction
df_ord = pd.read_parquet('../dataset/cleaned/orders.parquet')
df_item = pd.read_parquet('../dataset/cleaned/order_items.parquet')
df_ship = pd.read_parquet('../dataset/cleaned/shipments.parquet')
df_ret = pd.read_parquet('../dataset/cleaned/returns.parquet')
df_rev = pd.read_parquet('../dataset/cleaned/reviews.parquet')

# Analytical
df_sale = pd.read_parquet('../dataset/cleaned/sales.parquet')
#df_submit = pd.read_parquet('../dataset/sample_submission.csv')

# Operational
df_inv = pd.read_parquet('../dataset/cleaned/inventory.parquet')
df_web = pd.read_parquet('../dataset/cleaned/web_traffic.parquet')
print("Load data successfully")

Loading data...
Load data successfully


# Tạo feature và phân loại khách hàng

In [16]:
total_refund_per_order = df_ret.groupby('order_id')['refund_amount'].sum()
df_ord['refund_amount'] = df_ord['order_id'].map(total_refund_per_order).fillna(0.0)
display(df_ord.sample(3))

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,payment_value,installments,refund_amount
43300,55910,2013-03-19,52121,28547,cancelled,paypal,mobile,organic_search,12307.94,1 period,0.0
314463,405653,2016-06-21,73175,37146,cancelled,apple_pay,mobile,social_media,11853.69,12 periods,0.0
508104,655180,2019-03-26,94258,73110,delivered,credit_card,mobile,social_media,24078.96,3 periods,0.0


In [17]:
df_customer = df_cus.copy()

total_oder_per_cus = df_ord.groupby('customer_id')['order_id'].count()
total_return_oder_per_cus = df_ord[df_ord['order_status'] == 'returned'].groupby('customer_id')['order_id'].count()
total_cancelled_oder_per_cus = df_ord[df_ord['order_status'] == 'cancelled'].groupby('customer_id')['order_id'].count()
total_paid_per_cus = df_ord.groupby('customer_id')['payment_value'].sum()
total_refund_per_cus = df_ord.groupby('customer_id')['refund_amount'].sum()
total_refund_cancelled_per_cus = df_ord[df_ord['order_status'] == 'cancelled'].groupby('customer_id')['payment_value'].sum()

df_customer['total_oder'] = df_customer['customer_id'].map(total_oder_per_cus).fillna(0)
df_customer['return_oder'] = df_customer['customer_id'].map(total_return_oder_per_cus).fillna(0)
df_customer['cancel_oder'] = df_customer['customer_id'].map(total_cancelled_oder_per_cus).fillna(0)
df_customer['actual_oder'] = df_customer['total_oder'] - df_customer['return_oder'] - df_customer['cancel_oder']

df_customer['total_paid'] = df_customer['customer_id'].map(total_paid_per_cus).fillna(0)
df_customer['total_refund'] = df_customer['customer_id'].map(total_refund_per_cus).fillna(0)
df_customer['total_refund_cancel'] = df_customer['customer_id'].map(total_refund_cancelled_per_cus).fillna(0)
df_customer['actual_paid'] = df_customer['total_paid'] - df_customer['total_refund'] - df_customer['total_refund_cancel']

df_customer['return_cancel_rate'] = ((df_customer['return_oder'] + df_customer['cancel_oder']) / df_customer['total_oder']).fillna(-1.0)

display(df_customer.sample(3))

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel,total_oder,return_oder,cancel_oder,actual_oder,total_paid,total_refund,total_refund_cancel,actual_paid,return_cancel_rate
28797,37101,47978,Viet Tri,2022-06-29,Male,45-54,referral,13.0,1.0,0.0,12.0,352955.02,1019.93,0.0,351935.09,0.076923
7863,10110,14710,Nam Dinh,2020-10-19,Female,35-44,email_campaign,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,-1.000000
80409,103824,55751,Hoi An,2019-02-06,Female,18-24,paid_search,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,-1.000000


In [18]:
df_ord = df_ord.sort_values(by=['customer_id', 'order_date'])
df_ord['interval_prev_order'] = (
    df_ord.groupby('customer_id')['order_date']
    .diff()
    .dt.days
)
inter_order_gap = (
    df_ord.groupby('customer_id')['interval_prev_order']
    .apply(lambda x: x.iloc[1:].mean())
)
df_customer['avg_interval'] = df_customer['customer_id'].map(inter_order_gap).fillna(0.0)

first_order = df_ord.groupby('customer_id')['order_date'].first()
last_order = df_ord.groupby('customer_id')['order_date'].last()
df_customer['first_order_date'] = df_customer['customer_id'].map(first_order).fillna(df_customer['signup_date'])
df_customer['last_order_date'] = df_customer['customer_id'].map(last_order).fillna(df_customer['signup_date'])

current_date = pd.to_datetime('2022-12-31', dayfirst=True)
df_customer['recency_days'] = (current_date - df_customer['last_order_date']).dt.days
df_customer['tenure_days'] = (current_date - df_customer['first_order_date']).dt.days

df_customer = df_customer.drop(columns=['city', 'gender', 'age_group', 'acquisition_channel'])

In [19]:
df_customer.info()
display(df_customer.sample())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  string        
 1   zip                  121930 non-null  string        
 2   signup_date          121930 non-null  datetime64[ns]
 3   total_oder           121930 non-null  float64       
 4   return_oder          121930 non-null  float64       
 5   cancel_oder          121930 non-null  float64       
 6   actual_oder          121930 non-null  float64       
 7   total_paid           121930 non-null  float64       
 8   total_refund         121930 non-null  float64       
 9   total_refund_cancel  121930 non-null  float64       
 10  actual_paid          121930 non-null  float64       
 11  return_cancel_rate   121930 non-null  float64       
 12  avg_interval         121930 non-null  float64       
 13  first_order_da

,customer_id,zip,signup_date,total_oder,return_oder,cancel_oder,actual_oder,total_paid,total_refund,total_refund_cancel,actual_paid,return_cancel_rate,avg_interval,first_order_date,last_order_date,recency_days,tenure_days
99481,128509,77414,2016-09-12,14.0,2.0,0.0,12.0,501032.74,25920.24,0.0,475112.5,0.142857,203.307692,2013-03-18,2020-06-12,932,3575


In [20]:
six_months_ago = pd.to_datetime('2022-06-01', dayfirst=True)
high_paid_threshold = df_customer['actual_paid'].quantile(0.75)

conditions = [
    # 1. Non-buyer
    (df_customer['total_oder'] == 0),

    # 2. Bad quality
    (df_customer['return_cancel_rate'] >= 0.5) | 
    (df_customer['actual_paid'] < df_customer['total_paid'] * 0.5),

    # 3. New
    (df_customer['first_order_date'] >= six_months_ago),

    # 4. One_time
    (df_customer['actual_oder'] == 1),

    # 5. Old
    (df_customer['recency_days'] > 730),

    # 6. High-value loyal
    (df_customer['actual_paid'] >= high_paid_threshold) &
    (df_customer['recency_days'] <= df_customer['avg_interval']),

    # 7. At-risk
    (df_customer['actual_oder'] > 1) &
    (df_customer['recency_days'] > df_customer['avg_interval'] * 1.5),
]

choices = [
    'non_buyer',
    'low_quality',
    'new_customer',
    'one_time',
    'old_customer',
    'high_value_loyal',
    'at_risk'
]

df_customer['customer_segment'] = np.select(conditions, choices, default='loyal')

In [21]:
print(df_customer['customer_segment'].value_counts())
display(df_customer.sample(15))

customer_segment
non_buyer           31684
old_customer        27874
one_time            17998
loyal               13911
high_value_loyal    12343
low_quality          9612
at_risk              7394
new_customer         1114
Name: count, dtype: int64


,customer_id,zip,signup_date,total_oder,return_oder,cancel_oder,actual_oder,total_paid,total_refund,total_refund_cancel,actual_paid,return_cancel_rate,avg_interval,first_order_date,last_order_date,recency_days,tenure_days,customer_segment
9890,12751,10163,2021-02-08,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,-1.000000,0.000000,2021-02-08,2021-02-08,691,691,non_buyer
100931,130377,78563,2021-03-01,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,-1.000000,0.000000,2021-03-01,2021-03-01,670,670,non_buyer
42585,54894,30534,2018-01-27,14.0,0.0,0.0,14.0,470298.68,0.00,0.00,470298.68,0.000000,257.076923,2012-08-04,2021-09-28,459,3801,at_risk
3570,4584,19118,2021-10-01,2.0,0.0,0.0,2.0,18461.52,0.00,0.00,18461.52,0.000000,316.000000,2012-07-19,2013-05-31,3501,3817,old_customer
67722,87518,50076,2015-08-25,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,-1.000000,0.000000,2015-08-25,2015-08-25,2685,2685,non_buyer
19264,24855,45808,2019-04-30,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,-1.000000,0.000000,2019-04-30,2019-04-30,1341,1341,non_buyer
66409,85841,66092,2019-07-02,3.0,1.0,0.0,2.0,56103.63,6996.00,0.00,49107.63,0.333333,364.500000,2015-03-22,2017-03-20,2112,2841,old_customer
75156,97089,65754,2021-12-13,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,-1.000000,0.000000,2021-12-13,2021-12-13,383,383,non_buyer
48516,62627,29341,2015-03-30,14.0,0.0,1.0,13.0,335676.52,0.00,46513.17,289163.35,0.071429,276.461538,2012-08-09,2022-06-12,202,3796,high_value_loyal
54580,70471,42631,2015-11-22,1.0,0.0,0.0,1.0,19180.96,0.00,0.00,19180.96,0.000000,0.000000,2017-07-29,2017-07-29,1981,1981,one_time
